# Build a miniscope recording, one stage at a time

A 1-photon miniscope recording is the end of a physical chain: neurons fire →
calcium binds an indicator → light is emitted → optics blur and dim it → the
brain moves → a sensor counts noisy photons. The minian *analysis* pipeline runs
that chain **backwards** to recover the neurons. Here we run it **forwards** —
and we build it **together**, choosing the physics as we go.

Each stage has the same rhythm:

1. **Understand** — what this piece of physics is, and what to watch.
2. **Explore** — drag sliders and *see* the effect.
3. **Commit** — set the values you want, add the stage, and move on.

The recording grows as we add stages; by the end we have a full synthetic movie
*and* the exact ground truth behind it — positions, traces, footprints, motion,
and which cells are even physically recoverable. That truth is what Notebook 2
scores the real pipeline against.

> The interactive sliders need a **live kernel** — run this notebook (don't just
> read it). Run cells top to bottom; each stage uses the choices committed above
> it.

## Setup

The simulator is `minisim`. We also pull in `mediapy` (smooth inline video playback) and a couple of tiny plotting helpers. `mediapy` finds `ffmpeg` on your PATH automatically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from ipywidgets import FloatSlider, HBox, VBox
from IPython.display import display
import mediapy

from minisim import (
    Acquisition, Optics, ImageSensor, Tissue, Output, Spec, simulate,
    PlaceNeurons, CellActivity, CellOptics, Render,
    Neuropil, Bleaching, BrainMotion, Vignette, Leakage, Sensor,
)

# Interactive backend: each widget builds ONE figure and updates it in place as
# you drag a slider -- smooth (no flicker) and immune to VS Code's duplicate-
# output bug (no Output widget / re-display). If plots ever do duplicate after
# reopening the notebook, run Command Palette -> "Developer: Reload Window".
%matplotlib widget
plt.ioff()  # don't auto-show figures; we display each canvas once, then mutate it
plt.rcParams["image.cmap"] = "magma"


def play(movie, fps=20, height=280, title=None):
    # normalize a (frame, h, w) movie to [0,1] and show a looping inline video
    arr = np.asarray(movie, dtype=float)
    lo, hi = float(arr.min()), float(arr.max())
    mediapy.show_video((arr - lo) / (hi - lo + 1e-9), fps=fps, height=height, title=title, codec="h264")


def interactive_panel(sliders, draw, canvas, ncols=2):
    # Wire every slider to redraw the SAME persistent canvas in place. `draw`
    # reads the slider values itself and mutates the figure; we never re-display.
    for s in sliders.values():
        if hasattr(s, "continuous_update"):  # sliders only; some widgets lack this trait
            s.continuous_update = False
        s.style.description_width = "104px"
        s.layout.width = "340px"
    for s in sliders.values():
        s.observe(lambda _change: draw(), names="value")
    draw()
    vals = list(sliders.values())
    per = -(-len(vals) // ncols)  # ceil
    display(HBox([VBox(vals[i * per:(i + 1) * per]) for i in range(ncols)]))
    display(canvas)

## Our recording, as we build it

Two things carry forward as we go:

- **`acq`** — the *scope*: optics, sensor, tissue, frame rate, duration. We set
  this first (Stage 1).
- **`steps`** — the ordered list of construction stages. We start empty and
  **append one stage per section** as we commit to it.

`build()` simulates the recording from whatever stages we've committed so far;
`preview()` does the same on a small, fast FOV for responsive sliders. (`preview`
keeps the real pixel size and optics — so blur and brightness look *exactly*
right — and just images a smaller patch of tissue for a few seconds.)

In [ ]:
SEED = 7
steps = []          # we append one StepSpec per stage, as we commit to it

# acq is created in Stage 1; these helpers read whatever it currently is.
def build(until=None, extra=None):
    # full-resolution recording from the stages committed so far. `extra` is one
    # step to run on top of the committed ones without appending it (e.g. preview
    # the optics on the placed cells before optics is formally committed).
    trial = list(steps) + ([extra] if extra is not None else [])
    spec = Spec(acquisition=acq, seed=SEED, output=Output(save_intermediates=True), steps=trial)
    return simulate(spec, until=until)

def preview(extra=None, until=None, n_px=128, duration_s=3.0):
    # fast, small-FOV recording for slider exploration: same pixel size + optics,
    # just a smaller patch and fewer frames. `extra` is a step to try on top of
    # the committed ones without appending it.
    fast = acq.model_copy(update={
        "image_sensor": acq.image_sensor.model_copy(update={"n_px_height": n_px, "n_px_width": n_px}),
        "duration_s": duration_s,
    })
    trial = list(steps) + ([extra] if extra is not None else [])
    spec = Spec(acquisition=fast, seed=SEED, output=Output(save_intermediates=True), steps=trial)
    return simulate(spec, until=until)

## Stage 1 — How miniscope imaging works

Before committing to a specific scope, let's build intuition for the physics that
turns a fluorescent neuron into pixels. **This section is a sandbox:** it runs the
simulator's real optics, tissue, and sensor math on a few hand-placed cells, but
it is *completely independent* of the recording we build later — drag anything you
like, nothing here carries forward until we **commit a scope** at the bottom.

The image is shown in **green**, the way GCaMP fluorescence looks: a dark field
with bright cells.

**Left — the side view** (the picture you'd sketch on a whiteboard): the miniscope
at top, its **working distance** down to the focal plane, the **tissue** we image
through, and five cells spanning a range of depths around the focal plane.
**Right — what the image sensor sees.**

First, the **cell type** — a setting at the top of the next cell, not a slider.
`_MORPHOLOGY` chooses the GCaMP variant: **soma-targeted** GCaMP (SomaGCaMP /
riboGCaMP) confines the indicator to the cell body, so each cell is a clean blob;
**cytosolic** GCaMP (the usual GCaMP6/7/8, and the default here) also fills the
**proximal dendrites**, adding a few thin, fainter projections. Watch those
dendrites as you add tissue or push a cell off focus: the thin features are the
*first* thing scatter, defocus, and the noise floor erase, so usually only the
in-focus, shallow cell keeps them — exactly why resolving fine neurites through
tissue is so hard. (Change `_MORPHOLOGY` / `_N_DENDRITES` and re-run to compare.)

Then eight knobs, four families of physics:

- **Forming the image** — *NA* (light collection ∝ NA², so higher NA is mostly
  *brighter* here; its sharpening is tiny because diffraction is already
  *sub-pixel* — a miniscope is **pixel-limited**, not diffraction-limited),
  *magnification* (zooms the field of view) and *pixel pitch* (how coarsely the
  chip samples). These last two also set how much light lands on **each pixel**:
  a pixel collects flux over the object area it sees, `(pitch / magnification)²`,
  so higher mag *or* finer pitch spreads the same light over more pixels and dims
  every one of them — the **resolution-vs-SNR tradeoff**.
- **Depth & focus** — *tissue thickness* (how much tissue we image through: thin →
  bright & crisp, thick → dim & smeared by scatter + attenuation) and *focus* (the
  V4's tunable focus, **±150 µm**: a cell off the focal plane blurs, and one beyond
  reach can never be made sharp). The shaded band marks the **depth of field**,
  `≈ n·λ/NA²` — raise NA and watch it *narrow* (sharper but shallower focus). Note
  the asymmetry too: going *deeper* costs both focus *and* light, so deep cells
  fade fastest.
- **Field curvature** — *field curv*: a miniscope has no room for the lens elements
  that flatten the focal plane, so the in-focus surface is a shallow **bowl**, not a
  plane (the side view draws it): cells off the optical axis come into focus
  *shallower*, nearer the scope. The radius is large (real scopes ≈ 2–3 mm) so the
  bowl is gentle, but because the depth of field is only microns, even a small
  off-axis shift pushes edge cells out of focus — which is why the corners of a
  miniscope movie are softer and people keep their cells of interest centered. Dial
  the radius down to exaggerate it.
- **Detecting the image** — *exposure* and *read noise*: the sensor turns light
  into noisy counts, and a dim cell can sink below the noise floor. That "is it
  above the noise?" question is exactly what decides whether the analysis pipeline
  can ever recover a cell.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyArrowPatch, Rectangle
from scipy.ndimage import zoom
from minisim.steps.cell import degrade_footprint, neuron_footprint

# A DISCONNECTED sandbox: real simulator physics on five hand-placed cells,
# independent of the scope we commit below. GCaMP-like LUT (black -> green).
GCAMP = LinearSegmentedColormap.from_list("gcamp", ["#000000", "#00b140", "#b6ffb6"])
WD_UM = 700.0          # Miniscope V4 nominal front working distance
CHIP_UM = 900.0        # sandbox sensor chip extent
_SOMA_UM, _EMISSION_NM, _BLUR_PER_UM = 6.0, 525.0, 0.05
# Round-trip scatter: light is attenuated going in (excitation ~470 nm) and
# coming back out (emission ~525 nm); the two legs give an effective MFP ~50 um.
_MFP_EX_UM, _MFP_EM_UM = 90.0, 110.0
_N_TISSUE = 1.33        # tissue refractive index (DOF formula)
_PX_REF_UM = 5.0 / 4.0  # = default pitch/mag (1.25 um/px); per-pixel light ~ (px_um/this)^2

# Cell-type settings (not sliders -- edit here and re-run the cell to switch).
# Soma-targeted GCaMP confines signal to the body; cytosolic (the default here)
# also fills a few tapering proximal dendrites -- the thin features that scatter,
# defocus, and the noise floor erase first.
_MORPHOLOGY = "cytosolic"  # "soma" or "cytosolic"
_N_DENDRITES = 4
_DENDRITE_LEN_UM = 45.0    # proximal-dendrite length
_DENDRITE_WIDTH_UM = 3.0   # base width (tapers to a thread toward the tip)


def _dof_half_um(na):
    # in-focus half-depth; textbook DOF ~ n*lambda/NA^2 -> shrinks as NA rises
    return _N_TISSUE * (_EMISSION_NM / 1000.0) / na**2


# five cells spanning a depth band around the 700 um focal anchor, at distinct
# lateral spots so they never overlap. 1 = shallowest, 5 = deepest.
_CELLS = [
    {"name": "1", "y": -40.0, "x": -75.0, "offset": -50.0, "color": "#4daf4a"},
    {"name": "2", "y":  38.0, "x": -30.0, "offset": -25.0, "color": "#1f9e89"},
    {"name": "3", "y":  -8.0, "x":   8.0, "offset":   0.0, "color": "#377eb8"},
    {"name": "4", "y":  40.0, "x":  48.0, "offset":  27.0, "color": "#e6a700"},
    {"name": "5", "y": -34.0, "x":  80.0, "offset":  55.0, "color": "#e41a1c"},
]

# Generate each cell's sharp shape ONCE on a fixed fine um grid, so the shape is
# independent of the sensor sampling. Changing mag/pitch later only *rescales*
# these patches (same cells, zoomed -- never re-randomized); only editing the
# cell-type settings above rebuilds them.
# 0.5 um/px reference grid: finer than the sensor ever samples (object-space pixel
# ~1.25 um at the defaults) and on par with the diffraction-limited spot the optics
# can't beat anyway -- a 1p miniscope is pixel-limited, never diffraction-limited --
# so the reference holds the "true" shape without ever being the bottleneck.
_REF_PX_UM = 0.5
_PATCH_HALF_UM = 64.0  # half-extent of each cell's patch (covers soma + dendrites)
_ref_cache = {}        # (morphology, n_dendrites) -> list of fixed reference patches


def _ref_patches():
    key = (_MORPHOLOGY, _N_DENDRITES)
    if key not in _ref_cache:
        rng = np.random.default_rng(0)  # fixed seed: identical shapes every build
        n = int(round(2 * _PATCH_HALF_UM / _REF_PX_UM))
        c = (n - 1) / 2.0  # each cell sits at the center of its own patch
        _ref_cache[key] = [
            neuron_footprint(
                (n, n), (c, c), _SOMA_UM / _REF_PX_UM, 0.35, rng,
                morphology=_MORPHOLOGY, n_dendrites=_N_DENDRITES,
                dendrite_length_px=_DENDRITE_LEN_UM / _REF_PX_UM,
                dendrite_width_px=_DENDRITE_WIDTH_UM / _REF_PX_UM,
            )
            for _ in _CELLS
        ]
    return _ref_cache[key]


def _add_centered(canvas, patch, cy, cx):
    # Accumulate `patch` into `canvas` with the patch center aligned to (cy, cx),
    # clipping whatever falls off the canvas edge.
    ph, pw = patch.shape
    y0, x0 = int(round(cy - (ph - 1) / 2.0)), int(round(cx - (pw - 1) / 2.0))
    ys0, xs0 = max(y0, 0), max(x0, 0)
    ys1, xs1 = min(y0 + ph, canvas.shape[0]), min(x0 + pw, canvas.shape[1])
    if ys0 < ys1 and xs0 < xs1:
        canvas[ys0:ys1, xs0:xs1] += patch[ys0 - y0:ys1 - y0, xs0 - x0:xs1 - x0]


def _image_cells(na, magnification, pixel_pitch_um, tissue_thickness_um,
                 focus_offset_um, exposure, read_noise_e, field_curv_mm):
    # Build the five-cell scene with the simulator's real optics/tissue/sensor.
    optics = Optics(na=na, magnification=magnification, emission_nm=_EMISSION_NM,
                    field_curvature_radius_um=field_curv_mm * 1000.0)
    tissue = Tissue(scatter_mfp_excitation_um=_MFP_EX_UM, scatter_mfp_emission_um=_MFP_EM_UM,
                    scatter_blur_per_um=_BLUR_PER_UM)
    px_um = pixel_pitch_um / magnification
    n_px = int(np.clip(round(CHIP_UM / pixel_pitch_um), 48, 512))
    sensor = ImageSensor(n_px_height=n_px, n_px_width=n_px, pixel_pitch_um=pixel_pitch_um,
                         quantum_efficiency=0.7, read_noise_e=read_noise_e,
                         gain_adu_per_e=1.0, bit_depth=8)
    c = (n_px - 1) / 2.0
    focal_dist = WD_UM + focus_offset_um
    dof = _dof_half_um(na)
    optical = np.zeros((n_px, n_px))
    info = []
    for cell, patch in zip(_CELLS, _ref_patches()):
        dist = WD_UM + cell["offset"]                          # distance from scope
        path = max(tissue_thickness_um + cell["offset"], 0.0)  # tissue the light crosses
        # field curvature: off-axis cells focus shallower (no field flattener), so
        # each cell sees its own focal depth set by its radius from the optical axis
        r = np.hypot(cell["y"], cell["x"])
        focal_eff = focal_dist - optics.focal_curvature_shift_um(r)
        # the fixed shape, rescaled to the current pixel size (same cell, zoomed)
        sharp = np.clip(zoom(patch, _REF_PX_UM / px_um, order=1), 0.0, 1.0)
        cy, cx = c + cell["y"] / px_um, c + cell["x"] / px_um
        placed = np.zeros((n_px, n_px))
        _add_centered(placed, sharp, cy, cx)
        defocus = optics.defocus_sigma_um(dist, focal_eff)
        sigma_um = np.hypot(optics.diffraction_sigma_um,
                            np.hypot(tissue.scatter_sigma_um(path), defocus))
        gain = tissue.attenuation(path) * optics.collection_efficiency
        optical += degrade_footprint(placed, sigma_um / px_um, gain)
        info.append({**cell, "dist": dist, "in_focus": abs(dist - focal_eff) <= dof})
    # per-pixel light: a pixel integrates flux over its object area px_um^2 =
    # (pitch/mag)^2, so finer pitch OR higher mag dims each pixel (normalized to
    # the default px size, so the default state is unchanged).
    rng = np.random.default_rng(1)  # sensor shot/read noise (stable across redraws)
    counts = sensor.photons_to_counts(optical * exposure * (px_um / _PX_REF_UM) ** 2, rng)
    meta = dict(px_um=px_um, n_px=n_px, fov_um=n_px * px_um, focal_dist=focal_dist,
                surf=WD_UM - tissue_thickness_um, dof=dof, optics=optics)
    return counts, info, meta


# Build the figure ONCE; the fixed-range colorbar (0-255) is created once so it
# never accumulates when we redraw.
_fig1, (_ax1L, _ax1R) = plt.subplots(1, 2, figsize=(11.5, 4.8))
_fig1.subplots_adjust(left=0.07, right=0.87, wspace=0.30, top=0.84, bottom=0.12)
if hasattr(_fig1.canvas, "header_visible"):
    _fig1.canvas.header_visible = False
_cax1 = _fig1.add_axes([0.89, 0.12, 0.015, 0.72])
_sm1 = ScalarMappable(norm=Normalize(0, 255), cmap=GCAMP); _sm1.set_array([])
_fig1.colorbar(_sm1, cax=_cax1, label="ADC counts (8-bit)")

_sliders1 = dict(
    na=FloatSlider(min=0.1, max=0.6, step=0.02, value=0.30, description="NA"),
    magnification=FloatSlider(min=1.0, max=10.0, step=0.5, value=4.0, description="mag"),
    pixel_pitch_um=FloatSlider(min=1.0, max=8.0, step=0.2, value=5.0, description="pitch um"),
    tissue_thickness_um=FloatSlider(min=0.0, max=600.0, step=20.0, value=100.0, description="tissue um"),
    focus_offset_um=FloatSlider(min=-150.0, max=150.0, step=10.0, value=0.0, description="focus um"),
    exposure=FloatSlider(min=1000.0, max=20000.0, step=500.0, value=6000.0, description="exposure"),
    read_noise_e=FloatSlider(min=0.0, max=10.0, step=0.5, value=2.0, description="read noise"),
    field_curv_mm=FloatSlider(min=0.5, max=8.0, step=0.5, value=2.5, description="field curv mm"),
)


def explore_imaging():
    v = {k: s.value for k, s in _sliders1.items()}
    counts, info, meta = _image_cells(**v)
    axL, axR = _ax1L, _ax1R
    axL.clear(); axR.clear()
    # -- left: side view (depth = distance from scope, 0 at top) --
    axL.set_xlim(-150, 150); axL.set_ylim(-90, 900); axL.invert_yaxis()
    axL.add_patch(Rectangle((-55, -85), 110, 70, color="0.25"))
    axL.text(0, -50, "miniscope", color="w", ha="center", va="center", fontsize=9)
    axL.axhline(0, color="0.25", lw=1.5)
    surf = meta["surf"]
    axL.add_patch(Rectangle((-150, surf), 300, 900 - surf, color="#f0c9a8", alpha=0.55, zorder=0))
    axL.text(-145, surf + 22, "tissue", color="#a0522d", fontsize=9, va="top")
    axL.axhline(surf, color="#a0522d", ls=":", lw=1)
    axL.add_patch(FancyArrowPatch((-120, 0), (-120, WD_UM), arrowstyle="<->",
                                  mutation_scale=10, color="0.4", lw=1))
    axL.text(-128, WD_UM / 2, "WD 700 um", rotation=90, ha="right", va="center",
             color="0.4", fontsize=8)
    dof = meta["dof"]
    # field curvature: the in-focus surface is a shallow bowl (edges focus
    # shallower), not a flat plane. Draw its cross-section + the DOF band around it.
    xg = np.linspace(-150, 150, 121)
    opt = meta["optics"]
    fcurve = meta["focal_dist"] - np.array([opt.focal_curvature_shift_um(abs(x)) for x in xg])
    axL.fill_between(xg, fcurve - dof, fcurve + dof, color="#1f6fb2", alpha=0.16, zorder=1)
    axL.plot(xg, fcurve, color="#1f6fb2", ls="--", lw=1.4, zorder=2)
    axL.text(145, fcurve[-1], f"focal surface\n+/-{dof:.0f} um DOF", color="#1f6fb2",
             ha="right", va="top", fontsize=8)
    for ci in info:  # id number inside each dot (matches sensor panel); in-focus = white ring
        axL.scatter(ci["x"], ci["dist"], s=300, color=ci["color"], zorder=5,
                    edgecolor=("white" if ci["in_focus"] else "0.2"), linewidth=2.0)
        axL.text(ci["x"], ci["dist"], ci["name"], color="white", ha="center", va="center",
                 fontsize=9, weight="bold", zorder=6)
    axL.set(xlabel="lateral (um)", ylabel="distance from scope (um)",
            title="side view: scope -> tissue -> cells")
    # -- right: what the image sensor sees (colorbar is the fixed _cax1) --
    axR.imshow(counts, cmap=GCAMP, vmin=0, vmax=255, interpolation="nearest")
    for ci in info:
        cy = (meta["n_px"] - 1) / 2 + ci["y"] / meta["px_um"]
        cx = (meta["n_px"] - 1) / 2 + ci["x"] / meta["px_um"]
        if 0 <= cx < meta["n_px"] and 0 <= cy < meta["n_px"]:  # skip cells off the FOV
            axR.text(cx, cy - _SOMA_UM / meta["px_um"] - 4, ci["name"], color=ci["color"],
                     ha="center", fontsize=8, weight="bold")
    axR.set(title=f"image sensor: {meta['n_px']}x{meta['n_px']} px  |  {meta['px_um']:.2f} um/px"
                  f"  |  FOV {meta['fov_um']:.0f} um\nNA {v['na']:g}: brightness NA^2="
                  f"{meta['optics'].collection_efficiency:.3f}, diffraction sigma "
                  f"{meta['optics'].diffraction_sigma_um*1000:.0f} nm",
            xlabel="sensor px", ylabel="sensor px")
    _fig1.canvas.draw_idle()


interactive_panel(_sliders1, explore_imaging, _fig1.canvas)

**Commit the scope — the UCLA Miniscope V4.** Everything above was
intuition-building and is now behind us. Here we fix the *actual* scope the rest of
the notebook uses, independent of whatever you dragged in the sandbox. These are V4
numbers: a **Python480 CMOS** sensor at **608×608**, 4.8 µm pixels behind a ~3×
GRIN objective (**effective NA 0.30**) → a ~0.97 mm field at 1.6 µm/px, imaging
GCaMP, with a ~700 µm front working distance and ~**2.1 mm** field curvature. The
depth of field isn't a free knob — it's *derived from NA* (`"auto"`), since the
optics set it. The **focus** is also left `"auto"`: the V4's focus is tunable, so
rather than pin a fixed depth we let it settle where it **minimizes total defocus**
over the layer we place next — and because that calculation folds in the field
curvature (off-axis cells focus shallower), it lands a little deeper than the naive
middle of the layer, recovering more of the off-axis cells. Exactly what you do at
the rig: turn the focus knob until the most cells are sharp.

In [ ]:
# UCLA Miniscope V4. Two "auto" fields: depth of field is derived from NA
# (n*lambda/NA^2), and the focus minimizes total defocus over the placed layer
# (curvature-aware median effective depth) -- the V4 focus is tunable, so we don't
# pin it. front_working_distance_um is the V4 lens->focus spec (informational).
acq = Acquisition(
    fps=20.0,
    duration_s=90.0,
    focal_depth_in_tissue_um="auto",       # tunable V4 focus -> tracks placed layer (median depth)
    front_working_distance_um=700.0,       # V4 spec; does not affect the simulation
    optics=Optics(
        na=0.30,                           # V4 effective NA
        magnification=3.0,
        emission_nm=525.0,                 # GCaMP emission
        field_curvature_radius_um=2100.0,  # V4 Petzval radius ~2.1 mm (no field flattener)
    ),
    image_sensor=ImageSensor(n_px_height=608, n_px_width=608,  # Python480 CMOS
                             pixel_pitch_um=4.8, bit_depth=8),
    tissue=Tissue(scatter_mfp_excitation_um=90.0,  # round-trip scatter: 470 nm in
                  scatter_mfp_emission_um=110.0,   #                     525 nm out
                  scatter_blur_per_um=0.05),
)
print(f"FOV {acq.fov_um[0]:.0f} x {acq.fov_um[1]:.0f} um | {acq.pixel_size_um:.2f} um/px "
      f"| DOF +/-{acq.optics.resolved_depth_of_field_um:.1f} um | {acq.n_frames} frames "
      f"@ {acq.fps:.0f} fps | focus: {acq.focal_depth_in_tissue_um}")

## Stage 2 — Place the neurons

Now we drop cell bodies into the tissue volume: a lateral position `(x, y)` plus a
**depth** `z`. We don't choose *how many* directly — the count falls out of the
**volumetric density** (cells/mm³) and the imaged volume
(`round(density · FOV-area · thickness)`), so a thicker slab holds proportionally
more cells, just like a real chunk of tissue. (A strictly planar layer would have
zero thickness, so we floor it at one soma diameter — a single sheet of cell bodies
still has cells.)

Depth is the quiet protagonist of this whole notebook: it decides how blurred and
how dim each cell becomes, and ultimately whether it is recoverable at all. So this
section has two beats. **First the placement itself**, shown sharp and optics-free —
the true population in the tissue. **Then, at the end, we push that exact placement
through the V4 optics** to see what the scope makes of it (`A_planted → A_observed`).
That degradation is *purely spatial* — it depends only on each cell's depth and the
focus, not on any calcium activity — so we can apply it the moment the cells exist.

Start from a **preset**, then fine-tune:

- **CA1** — a thin pyramidal layer (~150 µm deep, near a single cell sheet).
- **Volumetric** — a thick 50–200 µm slab, with cells at every depth.

You also pick the **GCaMP variant** — *soma-targeted* (cell body only) or
*cytosolic* (soma + a few proximal dendrites). It does not move the distribution
dots (those mark cell bodies), but you will see it in the `A_planted → A_observed`
panels below: cytosolic's thin dendrites are precisely what the optics erase first.

Two views of the same population, at the **full committed FOV**:

- **Top** — looking straight down, each neuron a disk at its true soma size,
  **colored by depth** (the third placement axis). At realistic density the cell
  *bodies* stay mostly well separated — hundreds of cells the analysis must tell
  apart, with the occasional close pair that becomes a hard case. (Real crowding,
  where neighbors smear into one another, comes from the **optics** — which you see
  at the end of this section.)
- **Side** — the same lateral `x`, now against depth `z`: the slab the cells
  occupy, the structure the flat top view hides.

**Explore:** density, the depth band, soma size, and `min_distance` — a Poisson-disk
spacing floor. Raise it and the layout regularizes; high enough and the count
**caps out**, since you cannot pack past the spacing. This stage is **purely
spatial** — just where the cells sit. How brightly each one responds is biology we
set in Stage 3, and the noise that turns brightness into a signal-to-noise ratio
comes later still, from the optics and the sensor. Whatever you land on here is what
we commit and image below, so this is also where you come back to retune once you
have seen the finished recording.

In [ ]:
from ipywidgets import ToggleButtons
from matplotlib.collections import PatchCollection
from matplotlib.patches import Circle
from minisim.steps import sample_neurons

_DEPTH_MAX = 300.0  # fixed depth color/axis scale, so nothing jumps as you drag

# Stage 2 runs at the FULL committed FOV (no preview shrink) -- seeing the real
# field is the whole point of the distribution story. We call sample_neurons(),
# the *placement* half of place_neurons: it returns centers WITHOUT stamping
# footprints, so even ~hundreds of cells redraw instantly under a slider.
_FOV_H, _FOV_W = acq.fov_um

_fig2 = plt.figure(figsize=(9.5, 6.6))
if hasattr(_fig2.canvas, "header_visible"):
    _fig2.canvas.header_visible = False
_gs2 = _fig2.add_gridspec(2, 1, height_ratios=[3, 1.5],
                          hspace=0.32, left=0.1, right=0.86, top=0.92, bottom=0.1)
_ax2top = _fig2.add_subplot(_gs2[0, 0])
_ax2side = _fig2.add_subplot(_gs2[1, 0], sharex=_ax2top)
_cax2 = _fig2.add_axes([0.88, 0.56, 0.015, 0.34])  # depth colorbar
_sm2 = ScalarMappable(norm=Normalize(0, _DEPTH_MAX), cmap="viridis"); _sm2.set_array([])
_fig2.colorbar(_sm2, cax=_cax2, label="depth z (um)")

_sliders2 = dict(
    density_per_mm3=FloatSlider(min=2000, max=60000, step=1000, value=45000, description="density/mm3"),
    depth_lo=FloatSlider(min=0, max=250, step=5, value=140, description="depth lo um"),
    depth_hi=FloatSlider(min=0, max=300, step=5, value=160, description="depth hi um"),
    soma_radius_um=FloatSlider(min=2, max=12, step=0.5, value=5, description="soma r um"),
    min_distance_um=FloatSlider(min=0, max=40, step=2, value=0, description="min dist um"),
)

# Presets set the sliders to a scenario; you can fine-tune from there. CA1 = a thin
# pyramidal sheet; Volumetric = a thick slab spanning many depths. (The slider
# DEFAULTS above match CA1, so a clean run opens there.)
_PRESETS2 = {
    "CA1 (thin ~150um)":   dict(density_per_mm3=45000, depth_lo=140, depth_hi=160, soma_radius_um=5, min_distance_um=0),
    "Volumetric (50-200um)": dict(density_per_mm3=8000, depth_lo=50, depth_hi=200, soma_radius_um=6, min_distance_um=0),
}
_preset2 = ToggleButtons(options=list(_PRESETS2), value="CA1 (thin ~150um)", description="preset")


def _apply_preset2(change):
    for _k, _val in _PRESETS2[change["new"]].items():
        _sliders2[_k].value = _val


_preset2.observe(_apply_preset2, names="value")

# GCaMP targeting variant. Doesn't change the distribution dots (those mark cell
# bodies); its effect shows up in the A_planted -> A_observed panels below, where
# cytosolic's thin dendrites are the first thing the optics erase.
_morph2 = ToggleButtons(options=["soma", "cytosolic"], value="cytosolic", description="GCaMP")


def explore_placement():
    v = {k: s.value for k, s in _sliders2.items()}
    spec = PlaceNeurons(density_per_mm3=v["density_per_mm3"], soma_radius_um=v["soma_radius_um"],
                        depth_range_um=(v["depth_lo"], max(v["depth_hi"], v["depth_lo"])),
                        min_distance_um=v["min_distance_um"])
    centers = sample_neurons(spec, _FOV_H, _FOV_W, np.random.default_rng(SEED))
    centers = np.asarray(centers, dtype=float).reshape(-1, 3)
    z, y, x = (centers[:, 0], centers[:, 1], centers[:, 2]) if len(centers) else ([], [], [])
    r = v["soma_radius_um"]

    # -- top view: looking straight down. Each cell is a TRUE-radius disk (so
    #    crowding/overlap is honest), colored by depth.
    _ax2top.clear()
    if len(centers):
        pc = PatchCollection([Circle((xi, yi), r) for xi, yi in zip(x, y)],
                             cmap="viridis", norm=Normalize(0, _DEPTH_MAX), alpha=0.8)
        pc.set_array(z); pc.set_edgecolor("white"); pc.set_linewidth(0.2)
        _ax2top.add_collection(pc)
    _ax2top.set_xlim(0, _FOV_W); _ax2top.set_ylim(0, _FOV_H); _ax2top.invert_yaxis()
    _ax2top.set_aspect("equal")
    _ax2top.set(title=f"top view: {len(centers)} neurons over the {_FOV_W:.0f} x {_FOV_H:.0f} um FOV "
                      f"(color = depth)  |  GCaMP: {_morph2.value}", ylabel="y (um)")
    _ax2top.tick_params(labelbottom=False)

    # -- side view: same lateral x, now vs depth z (the axis the top view hides).
    #    Shaded band = the depth range you set. No focal plane (no optics yet).
    _ax2side.clear()
    _ax2side.axhspan(v["depth_lo"], max(v["depth_hi"], v["depth_lo"]), color="0.88", zorder=0)
    if len(centers):
        _ax2side.scatter(x, z, c=z, cmap="viridis", vmin=0, vmax=_DEPTH_MAX, s=9)
    _ax2side.set_xlim(0, _FOV_W); _ax2side.set_ylim(0, _DEPTH_MAX); _ax2side.invert_yaxis()
    _ax2side.set(title="side view: depth distribution", xlabel="x (um)", ylabel="depth z (um)")
    _fig2.canvas.draw_idle()


_morph2.observe(lambda _c: explore_placement(), names="value")  # redraw to update the title
display(HBox([_preset2, _morph2]))
interactive_panel(_sliders2, explore_placement, _fig2.canvas)

**Commit the placement.** We read the sliders above, so whatever you dialed in (preset + tweaks) is exactly what gets imaged. Re-running is idempotent — it *replaces* any previous placement rather than stacking a second one — so retune and re-run this as often as you like.

In [ ]:
_v2 = {k: s.value for k, s in _sliders2.items()}
steps[:] = [s for s in steps if s.kind != "place_neurons"]  # idempotent: drop any prior placement
steps.append(PlaceNeurons(
    density_per_mm3=_v2["density_per_mm3"], soma_radius_um=_v2["soma_radius_um"],
    depth_range_um=(_v2["depth_lo"], max(_v2["depth_hi"], _v2["depth_lo"])),
    min_distance_um=_v2["min_distance_um"], morphology=_morph2.value))
print(f"committed placement ({_morph2.value} GCaMP):", {k: round(float(x), 1) for k, x in _v2.items()})

### See it through the scope — `A_planted → A_observed`

The optical degradation is **purely spatial**: it depends on each cell's depth and
the focus, not on its calcium activity, so we apply it the moment the cells are
placed. Every sharp planted footprint is **dimmed** — light scatters and is absorbed
climbing back out of the tissue, and only the fraction set by NA² is collected — and
**blurred** by diffraction, by defocus for any cell off the focal surface, and by
scatter. The result, `A_observed`, is the best the analysis could ever recover; the
sharp `A_planted` is the answer key it will be graded against.

We focus on the layer we placed (`focal_depth_in_tissue_um="auto"` → the depth that
minimizes total defocus, which folds in the curvature below), just like turning the
focus knob at the rig. Two things then shape what survives: **depth** (cells far
from the focal depth blur and dim) and **field curvature** (the V4 has no
field-flattener, so its focal surface bows *shallower* toward the edges — even a
cell at exactly the right depth falls out of focus away from the optical axis; the
`"auto"` focus already leans deeper to claw some of them back). So for a **thin CA1
layer**, where depth is uniform, the
sharp cells form a central island that softens toward the rim — that is pure field
curvature, and it is why people keep their cells of interest near center. For the
**volumetric slab**, depth and curvature compound: only mid-depth, near-axis cells
stay crisp, shallow ones blur, and the deep ones fade toward nothing. Switch the
preset above, re-run the commit, and re-run this cell to compare — that is the
retuning loop in action.

> Each panel is scaled to its own peak with the GCaMP green LUT, so the faint
> observed field is lifted to full visibility: the top row is about *shape* (which
> cells stay sharp, which smear out), not brightness. The real dimming, observed
> peaks at only a few percent of planted, is reported in the title and bites later
> at the sensor noise floor. For the volumetric slab you can still read *which*
> cells survive, the in-focus mid-depth ones. The bottom pair zooms a single cell,
> also per-panel normalized, isolating the **blur** the optics adds on top.

In [ ]:
# Degradation is independent of calcium, so push the committed placement straight
# through the optics (place_neurons + optics; focus "auto" = median depth). This is
# the one slow build in Stage 2 -- it stamps and degrades a footprint per cell.
rec2 = build(until="optics", extra=CellOptics()); gt2 = rec2.ground_truth
A_pl, A_ob = gt2.A_planted, gt2.A_observed
peak_ratio = float(A_ob.max() / A_pl.max())
i_cell = int(np.argmax(A_ob.reshape(A_ob.shape[0], -1).max(axis=1)))  # brightest observed cell
cy, cx = gt2.centers_um[i_cell, 1:] / acq.pixel_size_um
# the resolved "auto" focus = median curvature-corrected effective depth (deeper
# than the median cell depth, because off-axis cells focus shallower).
_ay, _ax = A_ob.shape[1] * acq.pixel_size_um / 2.0, A_ob.shape[2] * acq.pixel_size_um / 2.0
_shift = np.array([acq.optics.focal_curvature_shift_um(rr)
                   for rr in np.hypot(gt2.centers_um[:, 1] - _ay, gt2.centers_um[:, 2] - _ax)])
_focal_um = float(np.median(gt2.depth_um + _shift))
# Each top panel is scaled to its own peak with the GCaMP green LUT, observed to its
# max footprint brightness, so the faint observed field is lifted to full visibility.
# The top row then reads as shape (which cells stay sharp), not brightness; the true
# dimming (observed peaks at a few percent of planted) is in the title and bites
# later at the sensor noise floor.


def _crop(a, cy, cx, half=22):
    h, w = a.shape
    return a[max(int(cy - half), 0):min(int(cy + half), h),
             max(int(cx - half), 0):min(int(cx + half), w)]


try:  # re-running this cell (the retuning loop) shouldn't pile up figures
    plt.close(_figO)
except NameError:
    pass
_figO = plt.figure(figsize=(10, 7.6))
if hasattr(_figO.canvas, "header_visible"):
    _figO.canvas.header_visible = False
_gsO = _figO.add_gridspec(2, 2, height_ratios=[3, 1.5], hspace=0.28, wspace=0.12,
                          left=0.04, right=0.96, top=0.9, bottom=0.04)
# top row: full FOV, each panel scaled to its own peak -> reads as shape, not brightness
for col, (img, ttl) in enumerate([
        (A_pl.max(0), "A_planted — ground truth (sharp, full brightness)"),
        (A_ob.max(0), f"A_observed — through the optics (peak {peak_ratio*100:.1f}% as bright)")]):
    ax = _figO.add_subplot(_gsO[0, col])
    ax.imshow(img, cmap=GCAMP, vmin=0, vmax=(img.max() or 1.0), origin="lower")
    ax.set_title(ttl, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
# bottom row: ONE cell up close, each re-normalized -> reveals the added blur
for col, (img, ttl) in enumerate([
        (_crop(A_pl[i_cell], cy, cx), "one cell, planted (sharp)"),
        (_crop(A_ob[i_cell], cy, cx), "same cell, observed (blurred + dimmed)")]):
    ax = _figO.add_subplot(_gsO[1, col])
    ax.imshow(img, cmap=GCAMP, vmin=0, vmax=(img.max() or 1.0), origin="lower")
    ax.set_title(ttl, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
_figO.suptitle(f"{gt2.n_units} cells ({_morph2.value} GCaMP) | focus auto @ {_focal_um:.0f} um "
               f"(deeper than the {np.median(gt2.depth_um):.0f} um median depth — field curvature) "
               f"| DOF +/-{acq.optics.resolved_depth_of_field_um:.1f} um "
               f"| in focus {int(gt2.in_focus.sum())}/{gt2.n_units}", fontsize=10)
display(_figO.canvas)

## Stage 3 — Calcium activity

Placement was about *where* cells are; this stage is about *when they fire* and the
fluorescence that follows. Each cell gets a spike train from a 2-state
(quiescent/active) gate driving a Poisson process, and every spike adds a calcium
transient with a fast rise and slow decay, `k(t) = e^{-t/τ_d} − e^{-t/τ_r}`. That
rise/decay shape is exactly what deconvolution later assumes.

Spikes are generated on a fine **~300 Hz** grid (one spike per ~3 ms bin respects the
refractory period), convolved with the kernel at that rate, then **binned down to the
camera frame rate** — exactly what the sensor's exposure does. So bursts can be dense
and physically tight even though we only ever store spikes at the frame rate.

Amplitude here is **biology**: a per-cell **brightness** gain (expression / response
strength) that scales a cell's *whole* trace, so some cells are simply brighter than
others. What is deliberately *not* here is **noise**: the trace you see is the clean
ground-truth `C`. Measurement noise is something the physical world adds later,
photon shot noise and read noise at the **sensor**, diffuse background at
**neuropil**, so a cell's signal-to-noise ratio is an *outcome* of the whole chain,
never a knob you set on the biology.

**Explore:**
- **peak t / FWHM** — the kernel's shape, in features you can read straight off a
  transient (time-to-peak, full width at half max); the simulator deduces `τ_r`/`τ_d`.
- **activity** — one sparse→dense knob (CaLab's spike-activity axis). Turning it up
  makes bursts start more often, last longer, and fire harder all at once, plus a
  little more background firing between them. 0.5 is CaLab's "moderate" default.
- **bright spread** — cell-to-cell brightness heterogeneity (a few bright cells over
  a dimmer majority).

(The sampling rule `τ_d · fps ≳ 1` must hold or the decay is unresolvable.)

In [ ]:
from minisim.steps import (calcium_kernel, spike_activity_params,
                                       tau_from_kernel_timing)

_fig3 = plt.figure(figsize=(10.5, 4.6))
if hasattr(_fig3.canvas, "header_visible"):
    _fig3.canvas.header_visible = False
_gs3 = _fig3.add_gridspec(2, 2, width_ratios=[3, 1], hspace=0.55, wspace=0.3,
                          left=0.07, right=0.97, top=0.9, bottom=0.14)
_ax3tr = _fig3.add_subplot(_gs3[:, 0])   # example traces + spike ticks
_ax3k = _fig3.add_subplot(_gs3[0, 1])    # the kernel shape
_ax3b = _fig3.add_subplot(_gs3[1, 1])    # per-cell brightness spread

# Kernel shape is set by observable features (time-to-peak, FWHM); the simulator
# inverts them to tau_r/tau_d. Defaults ~ GCaMP6f and reproduce tau_r~50ms, tau_d~0.5s.
_sliders3 = dict(
    kernel_peak=FloatSlider(min=0.04, max=0.40, step=0.01, value=0.13, description="peak t (s)"),
    kernel_fwhm=FloatSlider(min=0.15, max=1.50, step=0.05, value=0.50, description="FWHM (s)"),
    activity=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.5, description="activity"),
    brightness_cv=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.4, description="bright spread"),
)


def _activity_from_sliders(v):
    # observable kernel features -> tau_r/tau_d; one "activity" knob (sparse->dense)
    # moves all four CaLab Markov params together (see spike_activity_params).
    tau_r, tau_d = tau_from_kernel_timing(v["kernel_peak"], v["kernel_fwhm"])
    p_q2a, p_a2q, active_rate, quiescent_rate = spike_activity_params(v["activity"])
    return CellActivity(active_rate_hz=active_rate, quiescent_rate_hz=quiescent_rate,
                        tau_rise_s=tau_r, tau_decay_s=tau_d,
                        p_quiescent_to_active=p_q2a, p_active_to_quiescent=p_a2q,
                        brightness_cv=v["brightness_cv"]), tau_r, tau_d


def explore_activity():
    v = {k: s.value for k, s in _sliders3.items()}
    step3, tau_r, tau_d = _activity_from_sliders(v)
    rec = preview(extra=step3, until="cell_activity", duration_s=100.0)
    g = rec.ground_truth
    fps = rec.spec.acquisition.fps
    t = np.arange(rec.spec.acquisition.n_frames) / fps

    # -- example traces: a few of the busier cells, stacked. Each trace is scaled to
    #    its own peak so dense bursts don't blow up the axis and every lane is legible
    #    (per-cell brightness lives in the histogram panel, not here). Spike ticks
    #    below each show the bursts the gate makes.
    _ax3tr.clear()
    busiest = np.argsort(g.S.sum(axis=1))[-5:] if g.n_units else []
    for row, u in enumerate(busiest):
        c = g.C[u] - g.C[u].min()
        c = c / (c.max() or 1.0)  # scale to its own peak -> a clean unit-height lane
        off = row * 1.15
        _ax3tr.plot(t, c + off, lw=0.9)
        spk = np.where(g.S[u] > 0)[0]
        _ax3tr.plot(t[spk], np.full(spk.shape, off - 0.18), "|", color="k", ms=4)
    _ax3tr.set(title="calcium traces C (each scaled to its peak) + spikes S (ticks)",
               xlabel="time (s)", ylabel="cell (offset)")
    _ax3tr.set_yticks([])

    # -- the kernel: one transient's shape, drawn finely out to 5*tau_d so the full
    #    decay tail is visible regardless of the recording's frame rate. The dotted
    #    line marks the time-to-peak; the grey line is half max (the FWHM level).
    _ax3k.clear()
    _kfps = 200.0  # fine display sampling, independent of acq.fps
    kk = calcium_kernel(tau_r, tau_d, _kfps); tk = np.arange(len(kk)) / _kfps
    _ax3k.plot(tk, kk, color="#21a366", lw=1.5)
    _ax3k.axvline(v["kernel_peak"], color="0.5", ls=":", lw=0.9)
    _ax3k.axhline(0.5, color="0.8", lw=0.7, zorder=0)
    _ax3k.set_title(f"kernel\n(τr {tau_r*1000:.0f}, τd {tau_d*1000:.0f} ms)", fontsize=9)
    _ax3k.set_xlabel("t (s)"); _ax3k.set_yticks([0, 0.5, 1])

    # -- per-cell brightness gain: wide spread = a few bright cells over a dim
    #    majority; 0 = every cell equally bright.
    _ax3b.clear()
    if g.n_units:
        _ax3b.hist(g.amplitude_per_cell, bins=18, color="#4c72b0")
    _ax3b.set(title="per-cell brightness", xlabel="gain", ylabel="cells")
    _fig3.canvas.draw_idle()


interactive_panel(_sliders3, explore_activity, _fig3.canvas)

**Commit the activity.** Reads the sliders above, so the kernel shape, firing/burstiness, and brightness you dialed in are exactly what gets rendered. Idempotent: re-running *replaces* any prior activity.

In [ ]:
_v3 = {k: s.value for k, s in _sliders3.items()}
_step3, _tau_r3, _tau_d3 = _activity_from_sliders(_v3)
steps[:] = [s for s in steps if s.kind != "cell_activity"]  # idempotent: drop prior activity
steps.append(_step3)
print(f"committed activity: kernel tau_r {_tau_r3*1000:.0f}ms / tau_d {_tau_d3*1000:.0f}ms, "
      f"activity {_v3['activity']} (0=sparse, 0.5=moderate, 1=dense), "
      f"brightness_cv {_v3['brightness_cv']}")
print("stages committed so far:", " -> ".join(s.kind for s in steps))

---

### Next: the cells become a movie

So far we have cells with positions, depths, and traces — but no image yet. The
next stages turn them into a movie and then degrade it the way the physical world
does: **optics** (blur + dim by depth), **render** (the first actual video!),
then **background, bleaching, motion, illumination, and sensor noise**. Those
stages produce real video at full resolution, so we'll decide *there* how to
balance slider responsiveness against fidelity — stage by stage.

## Stage 4 - Render the movie

We finally have everything a frame needs: **where** each cell is and how the optics
degrade it (Stage 2, the `A_planted -> A_observed` reveal), and **when** it fires
(Stage 3, the trace `C`). Rendering just puts them together. Every cell contributes
its observed footprint, scaled frame by frame by its own calcium trace, and the
frame is the sum over all cells:

$$\text{movie}[t] \;=\; \sum_i A^{\text{obs}}_i \cdot C_i[t]$$

That product is the whole point of the module. A calcium movie is a low-rank
object: a handful of fixed spatial footprints `A`, each breathing in time by its
trace `C`. **CNMF, the analysis pipeline, exists to invert exactly this** - to
recover `A` and `C` from the movie. Here we run it forward, so we hold the true
`A` and `C` that any analysis is trying to get back.

This is the **first real video**, and it is deliberately *clean*: no background
haze, no bleaching, no motion, no sensor noise yet. Those degrade it in the stages
that follow, so this is the best-case image the pipeline could ever be handed.

A note on size: a full-resolution recording at the committed FOV and duration is a
multi-GB array. So the scrubber below renders a short window at full resolution
(responsive to drag), and a separate cell at the end exports a longer stretch as a
video. Drag **frame** to scrub time; drag **cell** to pick which cell's footprint
and trace are shown (white box = that cell in the frame).

In [ ]:
from ipywidgets import IntSlider

# Render ONCE: a short, full-resolution window. We apply the optics previewed in
# Stage 2 (CellOptics) and the render (Render) on top of the committed stages,
# without committing them yet -- the commit cell below does that.
_WINDOW_S = 8.0
_acq4 = acq.model_copy(update={"duration_s": _WINDOW_S})
_spec4 = Spec(acquisition=_acq4, seed=SEED, output=Output(save_intermediates=True),
              steps=list(steps) + [CellOptics(), Render()])
_rec4 = simulate(_spec4, until="cells_only")
_gt4 = _rec4.ground_truth
_mov4 = np.asarray(_rec4.observed, dtype=float)        # (frame, h, w), rendered once
_px4 = _rec4.spec.acquisition.pixel_size_um
_fps4 = _rec4.spec.acquisition.fps
_t4 = np.arange(_mov4.shape[0]) / _fps4
_vmax4 = float(np.percentile(_mov4, 99.9))             # shared ceiling so flicker shows
_H4, _W4 = _mov4.shape[1:]

# Cell picker list: the busiest cells first (detectable ones preferred), so the
# trace + footprint panels always land on something worth looking at.
_busy4 = np.argsort(_gt4.S.sum(axis=1))[::-1]
_cells4 = [int(u) for u in _busy4 if _gt4.detectable[u]][:24] or [int(u) for u in _busy4[:24]]
_peak0 = int(np.argmax(_gt4.C[_cells4[0]])) if _cells4 else 0

_fig4 = plt.figure(figsize=(11, 5))
if hasattr(_fig4.canvas, "header_visible"):
    _fig4.canvas.header_visible = False
_gs4 = _fig4.add_gridspec(2, 2, width_ratios=[2.4, 1], hspace=0.5, wspace=0.22,
                          left=0.04, right=0.97, top=0.9, bottom=0.13)
_ax4mov = _fig4.add_subplot(_gs4[:, 0])   # the rendered frame (scrubbable)
_ax4foot = _fig4.add_subplot(_gs4[0, 1])  # the picked cell's observed footprint
_ax4tr = _fig4.add_subplot(_gs4[1, 1])    # the picked cell's trace, with a cursor

_sliders4 = dict(
    cell=IntSlider(min=0, max=max(len(_cells4) - 1, 0), step=1, value=0, description="cell"),
    frame=IntSlider(min=0, max=_mov4.shape[0] - 1, step=1, value=_peak0, description="frame"),
)


def explore_render():
    u = _cells4[_sliders4["cell"].value] if _cells4 else 0
    k = _sliders4["frame"].value
    A_u = np.asarray(_gt4.A_observed[u]); C_u = np.asarray(_gt4.C[u])
    cy, cx = _gt4.centers_um[u, 1] / _px4, _gt4.centers_um[u, 2] / _px4
    z_u = _gt4.centers_um[u, 0]
    half = 55
    y0, y1 = max(int(cy) - half, 0), min(int(cy) + half, _H4)
    x0, x1 = max(int(cx) - half, 0), min(int(cx) + half, _W4)

    # left: the full frame at time k, with the picked cell boxed
    _ax4mov.clear()
    _ax4mov.imshow(_mov4[k], cmap=GCAMP, vmin=0, vmax=_vmax4)
    _ax4mov.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, ec="white", fc="none", lw=1.2))
    _ax4mov.set_title(f"movie frame {k}  (t = {_t4[k]:.1f} s)   =   sum_i  A_i . C_i[t]", fontsize=10)
    _ax4mov.set_xticks([]); _ax4mov.set_yticks([])

    # top-right: that cell's observed footprint (A), depth-blurred and dimmed
    _ax4foot.clear()
    _ax4foot.imshow(A_u[y0:y1, x0:x1], cmap=GCAMP)
    _ax4foot.set_title(f"A_observed (z = {z_u:.0f} um)", fontsize=9)
    _ax4foot.set_xticks([]); _ax4foot.set_yticks([])

    # bottom-right: that cell's trace (C), with the displayed frame marked
    _ax4tr.clear()
    _ax4tr.plot(_t4, C_u, color="#21a366", lw=1.0)
    _ax4tr.axvline(_t4[k], color="0.5", ls=":", lw=1.0)
    _ax4tr.set_title("C (this cell), cursor = frame", fontsize=9)
    _ax4tr.set_xlabel("time (s)"); _ax4tr.set_yticks([])
    _fig4.canvas.draw_idle()


interactive_panel(_sliders4, explore_render, _fig4.canvas)

**Commit the optics + render.** Locks in the depth blur/dimming previewed in Stage 2 and the movie compositing. Idempotent: re-running replaces any prior optics/render so the stage list never accumulates duplicates.

In [ ]:
steps[:] = [s for s in steps if s.kind not in ("optics", "render")]  # idempotent
steps.append(CellOptics())
steps.append(Render())
print("committed: optics (depth blur + dimming) -> render (the movie)")
print("stages committed so far:", " -> ".join(s.kind for s in steps))

### Export the full movie

The scrubber above used a short window so dragging stays responsive. This cell
renders a longer stretch at full resolution and plays it inline, so you can watch
the real dynamics. It is heavier (a multi-second render and a larger array), so it
is split out from the interactive panel. Lower `_EXPORT_S` if you are memory-bound,
or raise it toward the committed duration for the whole recording.

In [ ]:
_EXPORT_S = 20.0  # seconds to encode at full resolution; raise toward acq.duration_s if RAM allows
_acqX = acq.model_copy(update={"duration_s": _EXPORT_S})
_specX = Spec(acquisition=_acqX, seed=SEED, output=Output(save_intermediates=True), steps=list(steps))
_recX = simulate(_specX, until="cells_only")
play(np.asarray(_recX.observed), fps=acq.fps, title=f"rendered movie ({_EXPORT_S:.0f}s, cells_only)")

---

*(More stages will be added below: background haze, bleaching, motion, uneven
illumination, and finally the sensor noise that turns this clean movie into raw
camera counts.)*